In [0]:
s3_ext_location = 's3://awae-yellow-taxi-case/'
from pyspark.sql.functions import col, lit

In [0]:
print("--- Starting Silver Integration Layer Processing ---")

# 1. Read Yellow Taxi from Bronze, add metadata column, and preserve ALL original columns
path_bronze_yellow = f"{s3_ext_location}/bronze/yellow_tripdata_bronze"
df_yellow = spark.read.format("delta").load(path_bronze_yellow)

df_yellow_aligned = df_yellow.withColumn("taxi_type", lit("yellow"))

In [0]:
# 2. Read Green Taxi from Bronze, rename specific columns to match Yellow, and preserve ALL original columns
path_bronze_green = f"{s3_ext_location}/bronze/green_tripdata_bronze"
df_green = spark.read.format("delta").load(path_bronze_green)

df_green_aligned = df_green.withColumnRenamed("lpep_pickup_datetime", "tpep_pickup_datetime") \
                           .withColumnRenamed("lpep_dropoff_datetime", "tpep_dropoff_datetime") \
                           .withColumn("taxi_type", lit("green"))

In [0]:
# 3. Merge both data streams using unionByName with allowMissingColumns=True
# This retains ALL service-specific columns from both data sources, filling missing fields with nulls
print("Merging Yellow and Green datasets with full column retention into the Silver layer...")
df_silver_final = df_yellow_aligned.unionByName(df_green_aligned, allowMissingColumns=True)

In [0]:
# 4. Save the full un-filtered aligned data stream to Silver layer in Delta format
path_silver = f"{s3_ext_location}/silver/taxi_tripdata_silver"
print(f"Saving dataset to Silver layer (Delta): {path_silver}")
df_silver_final.write.format("delta").mode("overwrite").save(path_silver)

In [0]:
display(df_silver_final.groupBy("taxi_type").count())

In [0]:
# 4. Save consolidated stream to Silver layer in Delta format
path_silver = f"{s3_ext_location}/silver/taxi_tripdata_silver"
print(f"Saving unaggregated dataset to Silver layer (Delta): {path_silver}")
df_silver_final.write.format("delta").mode("overwrite").save(path_silver)

print("Silver integration layer successfully created!")

In [0]:
## Veryfing final schema
df_silver_final.printSchema()